# Soumission finale — JADCO / Collection Équinoxe 2026

## 1. Question économique et cible P1
Estimer la croissance same-unit annualisée du **loyer effectif fourni**, sur une cohorte définie avant 2026. Ce résultat décrit l’extrait et sa cohorte ; il ne mesure ni revenu total, ni occupation, ni hausse réglementaire universelle.

Pour chaque unité candidate, `p` estime la probabilité d’une transition qualifiée et `g` la croissance conditionnelle. La contribution prévue est `c = p × g` ; **P1 = 100 × médiane(c)**. La cible rétrospective attribue zéro aux unités sans transition et, aux autres, la moyenne des croissances annuelles qualifiées de l’unité. P1 diffère de la médiane descriptive de toutes les transitions. La médiane des produits n’est généralement pas le produit de leurs résumés.

L’annualisation est `(nouveau_effectif / ancien_effectif) ** (365,25 / jours_entre_débuts) − 1`. Les modules travaillent en fractions ; les résultats sont en pourcentage et les erreurs en points de pourcentage.

## 2. Chargement reproductible et confidentialité
Les quatre fichiers demeurent dans `data/raw/`. Les clés sont lues comme chaînes afin de préserver les zéros initiaux. Aucun enregistrement individuel n’est affiché ou exporté. Les sources externes sont chargées comme contexte documentaire ; le système gelé ne les utilise pas comme prédicteurs.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
assert (ROOT / 'notebooks/starter.ipynb').exists()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.models.final_submission import estimate_2026, backtest, production_summary, FROZEN_FINGERPRINT
from src.models.contextual_experts import rules_fingerprint
from src.evaluation.contextual_orchestration import run_contextual_orchestration
raw = ROOT / 'data/raw'
frames = {name: pd.read_csv(raw / f'equinoxe_{name}.csv',
           dtype={'sPropCode':'string','sUnitCode':'string'})
          for name in ['listings','lease_history','concessions','asking_history']}
leases, asking = frames['lease_history'], frames['asking_history']
external = pd.read_csv(ROOT / 'data/external/external_signals.csv')
display(pd.DataFrame([{'jeu':name,'lignes':len(df),'colonnes':df.shape[1]}
                      for name,df in frames.items()]))
assert rules_fingerprint() == FROZEN_FINGERPRINT

,jeu,lignes,colonnes
0,listings,1061,29
1,lease_history,4302,35
2,concessions,3560,18
3,asking_history,3857,14


## 3. Clé correcte, qualité et effet de composition
`UNIT_KEY = (sPropCode, sUnitCode)` est la clé officielle ; jamais `(sSite, sUnitCode)`. L’audit confirme 1 061 unités distinctes dans les listings et 248 lignes en collision avec la clé site/unité. Les quatre fichiers ne contiennent aucun doublon exact et aucune valeur manquante détectée par pandas ; cela ne garantit pas la validité économique des champs.

L’extrait comporte neuf codes de propriété et six bâtiments. Le changement de médiane contractuelle en 2023 (+10,85 %) mélange composition et prix : les nouveaux bâtiments et caractéristiques modifient les niveaux. Les transitions comparables suivent les mêmes unités ; aucun taux d’occupation ou d’absorption n’est inféré. Les durées atypiques et extrêmes n’ont pas été supprimés automatiquement.

## 4. Loyer effectif, concessions et renouvellement
Le champ `sRentEffective` est conservé comme convention économique source, sans reconstruction universelle des concessions. 88,22 % des lignes PromoPay portent `sMonths=1` : ce code n’est pas mensualisé sur tout le bail. L’audit économique a trouvé des détails incomplets et des désaccords avec `sConcession` ; ses hypothèses ne justifient pas une conversion universelle. Le contractuel demeure un diagnostic secondaire.

Renouvellements et relocations sont économiquement distincts ; leur statut **futur** n’est pas un prédicteur. Québec et Ontario restent distincts ; aucune règle TAL n’est appliquée à The Met et aucun plafond Ontario aux propriétés québécoises.

## 5. Same-unit et disponibilité à la coupure
Pour l’année Y, l’origine est le 31 décembre Y−1. Les baux connus ont déjà commencé et été signés ; la cohorte retient les unités dont le dernier bail connu expire en Y, selon les validations et exclusions d’ambiguïté de la fondation. Les unités ne sont pas sélectionnées selon leurs transitions futures.

Les labels d’entraînement sont qualifiés par la maturité : maximum des débuts, signatures et fins des deux baux, au plus tard à l’origine. Les cohortes historiques sont reconstruites à leur propre origine. Les labels de test sont révélés après prédiction. Les caractéristiques proviennent des baux connus, jamais des listings actuels ou du bail futur.

Réserve : le CRM est un extrait rétrospectif sans journal de versions. Les dates empêchent l’emploi des baux futurs mais ne prouvent pas l’absence de corrections historiques. Les backtests sont des simulations temporelles prudentes, conditionnelles à la fidélité source.

## 6. Architecture occurrence × croissance et experts gelés
**A — stabilité** : occurrence moyenne historique qualifiée et médiane des croissances positives au sens occurrence=1 ; les croissances négatives valides sont conservées.

**B — structure** : `Model B — full hierarchical` (`P1_G1`). Occurrence province → province × mois d’échéance (λ=20, minimum=1) ; croissance provinciale (λ=1, minimum=10). Les populations d’entraînement A/B restent celles déjà validées. B est la représentation complète, pas un champion sélectionné.

**C — régime récent si qualifié** : dernière cohorte calendaire Y−1 ; support minimum 30 par composante et couverture des labels ≥80 %. Les labels immatures restent inconnus. C peut être absent ; aucun seuil n’est abaissé en 2026.

## 7. Contexte, confiance et gel définitif
Confiance A=1. Pour B : `q=n/(n+30)`, `d=min(|B−A|, |B−global_B|)`, `t_B=q×d/(d+échelle)` ; confiance nulle en repli global. Pour C : qualification précédente, puis `t_C=q×couverture×cohérence×divergence/(divergence+échelle)` ; cohérence=`dispersion_longue/(dispersion_longue+dispersion_récente)`. Les poids A/B/C sont normalisés **séparément pour p et g**. L’échelle occurrence vient de A ; l’échelle croissance est le maximum de l’IQR historique, de |g_A| et de l’epsilon numérique.

Ces confiances sont des heuristiques de support et pertinence, pas des probabilités de précision. Aucun poids n’utilise les erreurs de l’année prédite. Le fichier du moteur, les configurations B et les constantes sont protégés par l’empreinte antérieure au scoring. Aucun réglage après observation de 2026.

## 8. Backtests 2023–2025 et expériences négatives
Les cinq références sont reproduites sans recherche de paramètres. A/B ont des erreurs P1 corrélées ; l’orchestration ne démontre pas un gain global. D, correction bâtiment, a été rejeté. Les variantes B plus simples ont parfois de meilleures métriques agrégées, sans désigner un champion sur trois années. Aucun signal fiable point-in-time n’a expliqué le ralentissement 2024 ni anticipé la rupture Saint-Elzear 2025. Ces résultats négatifs sont conservés.

In [2]:
historical = run_contextual_orchestration(leases, external, expected_fingerprint=FROZEN_FINGERPRINT)
display(historical['results'])
display(historical['metrics'])
# Interfaces obligatoires : prédictions avant révélation des labels de chaque année.
backtests = pd.DataFrame([backtest(leases, y) for y in (2023,2024,2025)])
display(backtests[['year','cohort_size','predicted_P1_pct','realized_P1_pct','error_pp']])
np.testing.assert_allclose(backtests.predicted_P1_pct,
                          historical['annual'].predicted_P1_pct, atol=1e-12)

,year,model,prediction_pct,realized_pct,error_pp,absolute_error_pp
0,2023,A,2.870060,2.036102,0.833957,0.833957
1,2023,Model B — full hierarchical,2.871314,2.036102,0.835212,0.835212
2,2023,A/B 50-50,2.870687,2.036102,0.834585,0.834585
3,2023,D,2.870927,2.036102,0.834825,0.834825
4,2024,A,3.269179,1.607852,1.661326,1.661326
5,2024,Model B — full hierarchical,3.236099,1.607852,1.628247,1.628247
6,2024,A/B 50-50,3.252639,1.607852,1.644787,1.644787
7,2024,D,3.289002,1.607852,1.681150,1.681150
8,2025,A,2.969328,3.300878,-0.331551,0.331551
9,2025,Model B — full hierarchical,2.924144,3.300878,-0.376735,0.376735


,model,MAE_pp,RMSE_pp,bias_pp,worst_absolute_error_pp
0,A,0.942278,1.090171,0.721244,1.661326
1,Model B — full hierarchical,0.946731,1.078687,0.695575,1.628247
2,A/B 50-50,0.944505,1.084323,0.708410,1.644787
3,D,0.949175,1.100473,0.728141,1.681150
4,ORCHESTRATED,0.942810,1.091772,0.724022,1.663993


,year,cohort_size,predicted_P1_pct,realized_P1_pct,error_pp
0,2023,405,2.872357,2.036102,0.836255
1,2024,645,3.271845,1.607852,1.663993
2,2025,856,2.972696,3.300878,-0.328182


## 9. Cohorte 2026 et prévision officielle
Coupure fixe : **2025-12-31**. Les échéances futures des baux déjà connus sont admissibles ; leurs futurs loyers réalisés ne le sont pas. Les diagnostics suivants sont uniquement agrégés par portefeuille, province et bâtiment. La croissance conditionnelle affichée est la médiane des `g` prédits sur **toutes les candidates** (aucun masque de transitions 2026 réalisées).

In [3]:
summary = production_summary(leases, asking, external)
forecast = estimate_2026(leases, asking, external)
assert summary['portfolio']['candidate_units'] == 931
assert np.isfinite(forecast) and forecast == summary['forecast_P1_pct']
print(f"P1 final 2026 : {forecast:.6f} %")
display(pd.DataFrame([summary['province_counts']], index=['Candidates par province']))
display(pd.Series(summary['expiry_month_counts'], name='Candidates par mois d’échéance').to_frame())
display(pd.DataFrame(summary['building_diagnostics']).T[['candidate_units','occurrence_pct',
          'conditional_growth_median_pct','median_contribution_pct']])
display(pd.DataFrame({k:summary['portfolio'][k] for k in ('occurrence','growth')}))
display(pd.DataFrame([summary['C_qualification']]))

P1 final 2026 : 2.445012 %


,Quebec,Ontario
Candidates par province,810,121


,Candidates par mois d’échéance
1,38
2,52
3,65
4,62
5,91
6,143
7,90
8,91
9,108
10,61


,candidate_units,occurrence_pct,conditional_growth_median_pct,median_contribution_pct
Le Carlyle,180,94.722007,2.566148,2.445164
Daniel-Johnson,128,94.79952,2.566148,2.445164
Levesque,75,94.605077,2.566148,2.445164
The Met,121,95.318876,2.511377,2.393582
Saint-Elzear,268,94.967923,2.566148,2.445164
Westpark,159,95.015726,2.566148,2.445164


,occurrence,growth
B_global_fallback_count,0.000000,0.000000
B_support_median,169.000000,2308.000000
B_support_min,1.000000,106.000000
B_support_max,356.000000,2308.000000
A_estimate_pct,95.236088,2.565881
A_mean_weight,0.926898,0.984360
A_min_weight,0.736231,0.922838
A_max_weight,0.996818,0.993551
A_activation_count,931.000000,931.000000
B_estimate_pct,95.038734,2.607293


,occurrence_qualified,growth_qualified,known_recent_labels,positive_recent_labels,recent_candidate_units,label_coverage,rule
0,False,False,89,6,856,0.103972,Support ≥30 par composante et couverture ≥80 %...


## 10. Scénarios défavorable / central / favorable
La méthode existante d’incertitude est réutilisée : 1 000 rééchantillonnages des contributions unitaires avec remplacement, seed=42. `E=max(|erreur P1 orchestrée|)` sur 2023/2024/2025. **LOW=q10(bootstrap)−E ; BASE=P1 gelé ; HIGH=q90(bootstrap)+E**.

Il s’agit de scénarios descriptifs de composition plus enveloppe empirique d’erreur. Ils ne corrigent pas le biais, ne garantissent aucune couverture et ne sont pas des intervalles de confiance calibrés. La dépendance entre unités et bâtiments n’est pas modélisée ; trois années ne suffisent pas à calibrer les risques de régime.

In [4]:
s = summary['scenarios']
display(pd.DataFrame([{'scénario':name,'P1_pct':s[name]} for name in ['LOW','BASE','HIGH']]))
print(f"Rayon historique : {s['model_error_radius_pp']:.6f} points de pourcentage")
assert s['LOW'] <= s['BASE'] <= s['HIGH']
# Seul artefact de données produit : résumé agrégé, JSON strict sans NaN.
output = ROOT / 'outputs/reports/final_2026_summary.json'
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(json.dumps(summary,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
assert rules_fingerprint() == FROZEN_FINGERPRINT

,scénario,P1_pct
0,LOW,0.781020
1,BASE,2.445012
2,HIGH,4.109156


Rayon historique : 1.663993 points de pourcentage


## 11. Pourquoi cette prévision ?
A apporte l’ancre historique ; B ajuste les probabilités d’occurrence et la croissance lorsqu’il possède un support segmenté et une différence informative. Les poids agrégés affichés montrent leur rôle réel, sans attribuer une causalité aux caractéristiques de bâtiment. C intervient uniquement si la dernière cohorte est assez mûre et supportée ; son statut et son support sont explicités ci-dessus.

La composition des 931 candidates influence la distribution des produits p×g et leur médiane. Les médianes par bâtiment et province sont des diagnostics, pas des composantes à sommer. Le taux attendu d’occurrence désigne les transitions qualifiées, **pas l’occupation**. Une valeur moyenne de p ne permet pas, à elle seule, de reconstruire P1.

## 12. Limites et interfaces exécutables
Seulement trois années de backtest ; erreurs A/B corrélées ; absence de signal fiable pour le ralentissement 2024 et Saint-Elzear 2025 ; C possiblement inactif ; scénarios non calibrés ; fidélité historique CRM conditionnelle ; population limitée à l’extrait et à la cohorte connue. La convention du loyer effectif source n’est pas une mesure auditée des encaissements. Les données SCHL, Statistique Canada, TAL et Ontario sont documentées dans le plan externe, mais aucune nouvelle source n’est qualifiée ni introduite dans ce système gelé.

Les fonctions importées **`estimate_2026(leases, asking, external=None)`** (pourcentage scalaire) et **`backtest(leases, target_year)`** (résumé agrégé) sont celles de la soumission et ont été exécutées ci-dessus. Le starter est préservé. Les fichiers bruts restent locaux et confidentiels ; aucun commit ou push n’est effectué.